# Validación del proyecto: fallas de SSD

Evalúa con validación cruzada el modelo final (el bosque con `min_samples_leaf=2`) y el segundo mejor del registro de MLflow (la red neuronal 32-16), con dos esquemas:

- **Estratificado (el que corresponde):** falla solo el 5,94 % de los SSD, así que cada pliegue tiene que tener la misma proporción de fallas. Cada SSD aparece una sola vez en los datos, así que no hace falta agrupar. Tampoco se puede validar por fecha: todos los sanos se midieron el 2019-12-31, y un corte por fecha dejaría pliegues sin sanos.
- **Al azar:** `KFold` mezclado, para comparar.

Después calcula una métrica de negocio: el costo de los errores con el umbral que lo minimiza.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

from pipeline_proyecto import COLUMNAS, OBJETIVO, cargar_datos, preparacion

datos = cargar_datos()
X = datos[COLUMNAS]
y = datos[OBJETIVO]
print(f"SSD: {len(datos)}  fallas: {int(y.sum())} ({y.mean():.2%})")

SSD: 176900  fallas: 10508 (5.94%)


In [2]:
MODELOS = {
    "bosque min_samples_leaf=2 (final)": lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=2, n_jobs=-1, random_state=42),
    "red neuronal 32-16 (segundo)": lambda: MLPClassifier(hidden_layer_sizes=(32, 16), early_stopping=True, max_iter=300, random_state=42),
}
ESQUEMAS = {
    "estratificado": StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    "al azar": KFold(n_splits=5, shuffle=True, random_state=42),
}

filas = []
for nombre_esquema, esquema in ESQUEMAS.items():
    for nombre_modelo, modelo in MODELOS.items():
        pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo())])
        inicio = time.perf_counter()
        puntajes = cross_validate(pipeline, X, y, cv=esquema, scoring="average_precision")["test_score"]
        filas.append({
            "esquema": nombre_esquema,
            "modelo": nombre_modelo,
            "ap_promedio": round(puntajes.mean(), 4),
            "desviacion": round(puntajes.std(), 4),
            "pliegues": [round(float(p), 4) for p in puntajes],
            "fallas_por_pliegue": [int(y.iloc[validacion].sum()) for _, validacion in esquema.split(X, y)],
        })
        print(f"{nombre_esquema:13s} {nombre_modelo:34s} AP {puntajes.mean():.4f} +- {puntajes.std():.4f}   ({time.perf_counter() - inicio:.0f} s)")

estratificado bosque min_samples_leaf=2 (final)  AP 0.5134 +- 0.0083   (37 s)


estratificado red neuronal 32-16 (segundo)       AP 0.3553 +- 0.0184   (43 s)


al azar       bosque min_samples_leaf=2 (final)  AP 0.5151 +- 0.0060   (21 s)


al azar       red neuronal 32-16 (segundo)       AP 0.3570 +- 0.0278   (60 s)


In [3]:
tabla = pd.DataFrame(filas)
tabla.to_csv("validacion_proyecto.csv", index=False)
print(tabla.to_string(index=False))

      esquema                            modelo  ap_promedio  desviacion                                 pliegues             fallas_por_pliegue
estratificado bosque min_samples_leaf=2 (final)       0.5134      0.0083   [0.526, 0.513, 0.5001, 0.5163, 0.5116] [2102, 2102, 2102, 2101, 2101]
estratificado      red neuronal 32-16 (segundo)       0.3553      0.0184 [0.3825, 0.3504, 0.3488, 0.3669, 0.3278] [2102, 2102, 2102, 2101, 2101]
      al azar bosque min_samples_leaf=2 (final)       0.5151      0.0060 [0.5086, 0.5121, 0.5205, 0.5106, 0.5239] [2078, 2064, 2066, 2149, 2151]
      al azar      red neuronal 32-16 (segundo)       0.3570      0.0278 [0.3258, 0.3273, 0.3949, 0.3559, 0.3811] [2078, 2064, 2066, 2149, 2151]


## La métrica del negocio

En este proyecto el error caro es el **falso negativo**: un SSD que falla sin aviso deja el servidor degradado, obliga a un reemplazo de urgencia y a reconstruir el arreglo, y arriesga datos. Lo estimo en **500 dólares** por falla no anticipada (técnico fuera de turno, horas de reconstrucción y riesgo de incumplir el SLA). El **falso positivo** cuesta mucho menos: revisar un disco sano, migrarle los datos y, en el peor caso, cambiarlo antes de tiempo. Lo estimo en **50 dólares**. Un fallo no anticipado cuesta 10 veces más que una falsa alarma.

Con las probabilidades de validación cruzada estratificada (cada SSD recibe la del modelo que no lo vio al entrenar), calculo el costo de cada umbral y me quedo con el más barato. La referencia es no hacer nada: todas las fallas llegan sin aviso.

In [4]:
COSTO_FALLA_SIN_AVISO = 500
COSTO_FALSA_ALARMA = 50
UMBRALES = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5]
estratificado = ESQUEMAS["estratificado"]

costo_sin_modelo = int(y.sum()) * COSTO_FALLA_SIN_AVISO
print(f"Sin modelo (todas las fallas sin aviso): {costo_sin_modelo:,} dolares")

negocio = []
for nombre_modelo, modelo in MODELOS.items():
    pipeline = Pipeline([("preparar", preparacion()), ("modelo", modelo())])
    probabilidades = cross_val_predict(pipeline, X, y, cv=estratificado, method="predict_proba")[:, 1]
    for umbral in UMBRALES:
        tn, fp, fn, tp = confusion_matrix(y, (probabilidades >= umbral).astype(int)).ravel()
        negocio.append({
            "modelo": nombre_modelo, "umbral": umbral,
            "fallas_anticipadas": tp, "fallas_sin_aviso": fn, "falsas_alarmas": fp,
            "recall": round(tp / (tp + fn), 4),
            "costo": fn * COSTO_FALLA_SIN_AVISO + fp * COSTO_FALSA_ALARMA,
        })

negocio = pd.DataFrame(negocio)
negocio["ahorro_vs_sin_modelo"] = (1 - negocio["costo"] / costo_sin_modelo).round(4)
negocio.to_csv("costo_proyecto.csv", index=False)
print(negocio.to_string(index=False))

Sin modelo (todas las fallas sin aviso): 5,254,000 dolares


                           modelo  umbral  fallas_anticipadas  fallas_sin_aviso  falsas_alarmas  recall   costo  ahorro_vs_sin_modelo
bosque min_samples_leaf=2 (final)    0.02                9508              1000           73904  0.9048 4195200                0.2015
bosque min_samples_leaf=2 (final)    0.05                8471              2037           40921  0.8061 3064550                0.4167
bosque min_samples_leaf=2 (final)    0.10                7196              3312           19477  0.6848 2629850                0.4995
bosque min_samples_leaf=2 (final)    0.15                6184              4324           10891  0.5885 2706550                0.4849
bosque min_samples_leaf=2 (final)    0.20                5329              5179            6494  0.5071 2914200                0.4453
bosque min_samples_leaf=2 (final)    0.30                3985              6523            2530  0.3792 3388000                0.3552
bosque min_samples_leaf=2 (final)    0.40                3141 

In [5]:
mejores = negocio.loc[negocio.groupby("modelo")["costo"].idxmin()]
print("El umbral mas barato de cada modelo:")
print(mejores.to_string(index=False))

El umbral mas barato de cada modelo:
                           modelo  umbral  fallas_anticipadas  fallas_sin_aviso  falsas_alarmas  recall   costo  ahorro_vs_sin_modelo
bosque min_samples_leaf=2 (final)     0.1                7196              3312           19477  0.6848 2629850                0.4995
     red neuronal 32-16 (segundo)     0.1                5925              4583           22049  0.5639 3393950                0.3540
